In [1]:
import pandas as pd

### Inisialisasi Dataset

In [ ]:
df_tiketcom = pd.read_csv('dataset/data_tiketcom_3600_reviews.csv')
df_traveloka = pd.read_csv('dataset/data_traveloka_3600_reviews.csv')

df_tiketcom['source'] = 'tiketcom'
df_traveloka['source'] = 'traveloka'

df = pd.concat([df_tiketcom, df_traveloka], ignore_index=True)

### Validasi awal dan data understanding

In [3]:
df.dtypes

reviewId                object
userName                object
userImage               object
content                 object
score                    int64
thumbsUpCount            int64
reviewCreatedVersion    object
at                      object
replyContent            object
repliedAt               object
appVersion              object
source                  object
dtype: object

In [4]:
use_cols = ['reviewId', 'content', 'score', 'at', 'source']
df = df[use_cols]
df.head()

,reviewId,content,score,at,source
0,d84a4955-4bfb-448b-a260-b36935a58c9b,kalo ada pembatalan hub langsung WA dong,1,2026-01-13 12:12:06,tiketcom
1,0b67f755-1fb1-4300-a769-68e45a0a48cc,ganguan terus susah cari jadwal disni,2,2026-01-13 11:18:17,tiketcom
2,fcef9cb8-5c08-41cf-a8b5-2a2eb1edfba8,banyakin promonya ya,5,2026-01-13 00:01:23,tiketcom
3,d039ee95-c3f2-4e76-ae03-9214f7672cd8,very bad service!!!!!!,1,2026-01-12 21:29:45,tiketcom
4,0d830287-f7df-454c-9598-4d6a6071b32a,mantap,5,2026-01-12 13:02:48,tiketcom


In [5]:
df.isnull().sum()

reviewId    0
content     0
score       0
at          0
source      0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [ ]:
# df.to_csv('data_all_7200_reviews.csv', index=False)
print("Data berhasil disimpan ke 'data_all_7200_reviews.csv'")

Data berhasil disimpan ke 'data_all_7200_reviews.csv'


### Casefolding dan Text Cleaned 

In [16]:
import re

df = pd.read_csv('dataset/data_all_7200_reviews.csv')
df_stopwords = pd.read_csv('dataset/data_text_informal_to_formal.csv', sep=';')

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()                             # ubah ke huruf kecil
    text = re.sub(r'http\S+|www\.\S+', '', text)    # hapus URL
    text = re.sub(r'@\w+', '', text)                # hapus mention (@username)
    text = re.sub(r'#\w+', '', text)                # hapus hashtag (#hashtag)
    text = re.sub(r'[^a-z\s]', '', text)            # hapus karakter non-huruf
    text = re.sub(r'\s+', ' ', text).strip()        # hapus spasi berlebih
    return text

df['cleaned_content'] = df['content'].apply(clean_text)
df =df[df['cleaned_content'].ne("")]

df['score'] = pd.to_numeric(df['score'], errors='coerce')
df = df[df['score'].between(1, 5)]

df = df.drop_duplicates(subset=['reviewId']).reset_index(drop=True)
df.head(5)

,reviewId,content,score,at,source,cleaned_content
0,d84a4955-4bfb-448b-a260-b36935a58c9b,kalo ada pembatalan hub langsung WA dong,1,2026-01-13 12:12:06,tiketcom,kalo ada pembatalan hub langsung wa dong
1,0b67f755-1fb1-4300-a769-68e45a0a48cc,ganguan terus susah cari jadwal disni,2,2026-01-13 11:18:17,tiketcom,ganguan terus susah cari jadwal disni
2,fcef9cb8-5c08-41cf-a8b5-2a2eb1edfba8,banyakin promonya ya,5,2026-01-13 00:01:23,tiketcom,banyakin promonya ya
3,d039ee95-c3f2-4e76-ae03-9214f7672cd8,very bad service!!!!!!,1,2026-01-12 21:29:45,tiketcom,very bad service
4,0d830287-f7df-454c-9598-4d6a6071b32a,mantap,5,2026-01-12 13:02:48,tiketcom,mantap


### Preprocessing Data Sentiment

#### (Handling Chatwords dan Stopword, Tokenizer, Stemming, TF-IDF)

In [ ]:
import re
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
#Preparing Chatword Dictionary
if "original-for" in df_stopwords.columns:
    df_stopwords = df_stopwords.rename(columns={"original-for": "original_for"})

chatword_dict = (
    df_stopwords
    .dropna(subset=["transformed", "original_for"])
    .assign(transformed=lambda x: x["transformed"].astype(str).str.strip().str.lower(),
            original_for=lambda x: x["original_for"].astype(str).str.strip().str.lower())
    .set_index("transformed")["original_for"]
    .to_dict()
)

In [44]:
# regex untuk menangkap token kata
token_pat = re.compile(r"\b\w+\b", flags=re.UNICODE)

def replace_chatwords(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()

    def repl(match):
        w = match.group(0)
        return chatword_dict.get(w, w)

    return token_pat.sub(repl, text)

df["normalized_text"] = df["cleaned_content"].apply(replace_chatwords)
df.head()

,reviewId,content,score,at,source,cleaned_content,normalized_text,no_stop_text,tokenized_text,stemmed_tokens,stemmed_text,label,len_words
0,d84a4955-4bfb-448b-a260-b36935a58c9b,kalo ada pembatalan hub langsung WA dong,1,2026-01-13 12:12:06,tiketcom,kalo ada pembatalan hub langsung wa dong,kalau ada pembatalan hub langsung wa dong,kalau pembatalan hub langsung wa dong,"[kalau, pembatalan, hub, langsung, wa, dong]","[kalau, batal, hub, langsung, wa, dong]",kalau batal hub langsung wa dong,negatif,6
1,0b67f755-1fb1-4300-a769-68e45a0a48cc,ganguan terus susah cari jadwal disni,2,2026-01-13 11:18:17,tiketcom,ganguan terus susah cari jadwal disni,ganguan terus susah cari jadwal disni,ganguan terus susah cari jadwal disni,"[ganguan, terus, susah, cari, jadwal, disni]","[ganguan, terus, susah, cari, jadwal, disni]",ganguan terus susah cari jadwal disni,negatif,6
2,fcef9cb8-5c08-41cf-a8b5-2a2eb1edfba8,banyakin promonya ya,5,2026-01-13 00:01:23,tiketcom,banyakin promonya ya,dibanyaki promonya ya,dibanyaki promonya,"[dibanyaki, promonya]","[banyak, promonya]",banyak promonya,positif,2
3,d039ee95-c3f2-4e76-ae03-9214f7672cd8,very bad service!!!!!!,1,2026-01-12 21:29:45,tiketcom,very bad service,very bad service,very bad service,"[very, bad, service]","[very, bad, service]",very bad service,negatif,3
4,0d830287-f7df-454c-9598-4d6a6071b32a,mantap,5,2026-01-12 13:02:48,tiketcom,mantap,mantap,mantap,[mantap],[mantap],mantap,positif,1


In [ ]:
# Remove Stopwords
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

df["no_stop_text"] = df["normalized_text"].apply(stopword_remover.remove)
df.head()

,reviewId,content,score,at,source,cleaned_content,normalized_text,no_stop_text
0,d84a4955-4bfb-448b-a260-b36935a58c9b,kalo ada pembatalan hub langsung WA dong,1,2026-01-13 12:12:06,tiketcom,kalo ada pembatalan hub langsung wa dong,kalau ada pembatalan hub langsung wa dong,kalau pembatalan hub langsung wa dong
1,0b67f755-1fb1-4300-a769-68e45a0a48cc,ganguan terus susah cari jadwal disni,2,2026-01-13 11:18:17,tiketcom,ganguan terus susah cari jadwal disni,ganguan terus susah cari jadwal disni,ganguan terus susah cari jadwal disni
2,fcef9cb8-5c08-41cf-a8b5-2a2eb1edfba8,banyakin promonya ya,5,2026-01-13 00:01:23,tiketcom,banyakin promonya ya,dibanyaki promonya ya,dibanyaki promonya
3,d039ee95-c3f2-4e76-ae03-9214f7672cd8,very bad service!!!!!!,1,2026-01-12 21:29:45,tiketcom,very bad service,very bad service,very bad service
4,0d830287-f7df-454c-9598-4d6a6071b32a,mantap,5,2026-01-12 13:02:48,tiketcom,mantap,mantap,mantap


#### (Handling Chatwords dan Stopword, Tokenizer, Stemming)

In [29]:
# Tokenization
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
df["tokenized_text"] = df["no_stop_text"].apply(word_tokenize)
df.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Kinan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,reviewId,content,score,at,source,cleaned_content,normalized_text,no_stop_text,tokenized_text
0,d84a4955-4bfb-448b-a260-b36935a58c9b,kalo ada pembatalan hub langsung WA dong,1,2026-01-13 12:12:06,tiketcom,kalo ada pembatalan hub langsung wa dong,kalau ada pembatalan hub langsung wa dong,kalau pembatalan hub langsung wa dong,"[kalau, pembatalan, hub, langsung, wa, dong]"
1,0b67f755-1fb1-4300-a769-68e45a0a48cc,ganguan terus susah cari jadwal disni,2,2026-01-13 11:18:17,tiketcom,ganguan terus susah cari jadwal disni,ganguan terus susah cari jadwal disni,ganguan terus susah cari jadwal disni,"[ganguan, terus, susah, cari, jadwal, disni]"
2,fcef9cb8-5c08-41cf-a8b5-2a2eb1edfba8,banyakin promonya ya,5,2026-01-13 00:01:23,tiketcom,banyakin promonya ya,dibanyaki promonya ya,dibanyaki promonya,"[dibanyaki, promonya]"
3,d039ee95-c3f2-4e76-ae03-9214f7672cd8,very bad service!!!!!!,1,2026-01-12 21:29:45,tiketcom,very bad service,very bad service,very bad service,"[very, bad, service]"
4,0d830287-f7df-454c-9598-4d6a6071b32a,mantap,5,2026-01-12 13:02:48,tiketcom,mantap,mantap,mantap,[mantap]


In [ ]:
# Stemming
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def stem_tokens(tokens):
    if not isinstance(tokens, list):
        return []
    return [stemmer.stem(t) for t in tokens]

df["stemmed_tokens"] = df["tokenized_text"].apply(stem_tokens)
df["stemmed_text"] = df["stemmed_tokens"].apply(lambda toks: " ".join(toks))

In [31]:
df.head()

,reviewId,content,score,at,source,cleaned_content,normalized_text,no_stop_text,tokenized_text,stemmed_tokens,stemmed_text
0,d84a4955-4bfb-448b-a260-b36935a58c9b,kalo ada pembatalan hub langsung WA dong,1,2026-01-13 12:12:06,tiketcom,kalo ada pembatalan hub langsung wa dong,kalau ada pembatalan hub langsung wa dong,kalau pembatalan hub langsung wa dong,"[kalau, pembatalan, hub, langsung, wa, dong]","[kalau, batal, hub, langsung, wa, dong]",kalau batal hub langsung wa dong
1,0b67f755-1fb1-4300-a769-68e45a0a48cc,ganguan terus susah cari jadwal disni,2,2026-01-13 11:18:17,tiketcom,ganguan terus susah cari jadwal disni,ganguan terus susah cari jadwal disni,ganguan terus susah cari jadwal disni,"[ganguan, terus, susah, cari, jadwal, disni]","[ganguan, terus, susah, cari, jadwal, disni]",ganguan terus susah cari jadwal disni
2,fcef9cb8-5c08-41cf-a8b5-2a2eb1edfba8,banyakin promonya ya,5,2026-01-13 00:01:23,tiketcom,banyakin promonya ya,dibanyaki promonya ya,dibanyaki promonya,"[dibanyaki, promonya]","[banyak, promonya]",banyak promonya
3,d039ee95-c3f2-4e76-ae03-9214f7672cd8,very bad service!!!!!!,1,2026-01-12 21:29:45,tiketcom,very bad service,very bad service,very bad service,"[very, bad, service]","[very, bad, service]",very bad service
4,0d830287-f7df-454c-9598-4d6a6071b32a,mantap,5,2026-01-12 13:02:48,tiketcom,mantap,mantap,mantap,[mantap],[mantap],mantap


In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,     
    ngram_range=(1, 2),      
    min_df=2                 
)

X_tfidf = tfidf.fit_transform(df["stemmed_text"])
print(X_tfidf.shape)

(7133, 5562)


In [49]:
df.to_csv('data_ready_for_modeling.csv', index=False)
print("Data berhasil disimpan ke 'data_ready_for_modeling.csv'")

Data berhasil disimpan ke 'data_ready_for_modeling.csv'
